In [1]:
import os
import json
import torch
import torch.nn.functional as F
import onnx
from onnx import numpy_helper
import numpy as np

def _weight_entropy(weight_matrix, epsilon=0.01):
	w_flat = weight_matrix.view(-1).float()
	size_w = w_flat.numel()
	p = F.softmax(w_flat, dim=0)
	entropy = -torch.sum(p * torch.log(p + epsilon)).item()
	return entropy, size_w

def _block_ewq_score(weight_list, epsilon=0.01):
	total_weighted = 0.0
	total_params = 0

	for W in weight_list:
		if isinstance(W, torch.nn.Parameter):
			W = W.data
		h_wi, size_wi = _weight_entropy(W, epsilon)
		total_weighted += h_wi / size_wi # fixed as * not /
		
		total_params += size_wi
		print(size_wi)
		
	print(total_weighted)
	print(total_params)
	if total_params > 0:
		return total_weighted / total_params
	return 0.0



In [2]:
def _normalize_scores(scores):
	arr = np.array(scores, dtype=np.float64)
	min_val = arr.min()
	max_val = arr.max()
	normalized = (arr - min_val) / (max_val + 1e-8)
	return normalized.tolist()


def _rank_layers(scores):
	arr = np.array(scores)
	ranked = np.argsort(arr).tolist()
	return ranked


def _build_output(metric_name, model_path, scores, extra=None):
	model_name = os.path.basename(model_path.rstrip("/\\"))
	num_layers = len(scores)
	layer_labels = [f"layer_{i}" for i in range(1, num_layers + 1)]
	layer_map = {label: float(s) for label, s in zip(layer_labels, scores)}
	out = {
		"metric": metric_name,
		"model": model_name,
		"num_layers": num_layers,
		"layer_scores": layer_map,
		"ranked_layers": _rank_layers(scores),
		"scores": scores,
	}
	if extra:
		out.update(extra)
	return out

def _get_sensitive_nodes():
	nodes = []
	
	return nodes

def _save_output(output_dict, output_dir, metric_name, model_path):
	model_name = os.path.basename(model_path.rstrip("/\\"))
	os.makedirs(output_dir, exist_ok=True)
	out_path = os.path.join(output_dir, f"{model_name}_{metric_name}.json")
	with open(out_path, "w") as f:
		json.dump(output_dict, f, indent=2)
	return out_path

In [3]:
import re


class ewq():
	def __init__(self, model_path="yolo26n-seg.onnx", target=0.35):
		self.model_path = model_path
		self.target = target

	def filter(self):
		arr = []
		threshold = 1.0
		layer_scores = []
		layer_names = []
		weight_params = {}
		section_weights = {} # eg. "head": [weight1, weight2, ...]
		section_names = {"backbone": 10, "neck":22, "head": 23}
		section_scores = {}

		module_weights = {}
		# module_names = [i for i in range(24)]
		module_scores = {}

		model = onnx.load(self.model_path)
		inits = {init.name: init for init in model.graph.initializer}  # Get the initializers of the original modelq
		_idx = 0
		for init in model.graph.initializer:
			 # debugging
			if "weight" in init.name and "conv" in init.name:
				# print("\n",init.name)
				layer_names.append(init.name)
				total_params = 0
				W_np = numpy_helper.to_array(inits[init.name])
				weights = torch.from_numpy(W_np).float()
				w_flat = weights.view(-1).float()
				size_w = w_flat.numel()
				weight_params[init.name] = size_w
				score = _block_ewq_score([weights])
				# print(score)
				### cal layer ewq score
				layer_scores.append(score)


				# remove layer score 
				# layer_scores.append(0)
				# print(layer_scores)

				## cal section ewq score
				match = re.match(r'model\.(\d+)', init.name)
				layer_idx = int(match.group(1))
				for section, idx in section_names.items():
					if layer_idx <= idx:
						if section not in section_weights.keys():
							section_weights[section] = []
						section_weights[section].append(weights)
						# print(section, layer_idx, idx, init.name)
						break

				### module ewq score
				if layer_idx not in module_weights.keys():
					module_weights[layer_idx] = []
				module_weights[layer_idx].append(weights)
				# print(f"Module {layer_idx}: weight {init.name} added.")

				_idx+=1
				# print(f"{_idx}: Section weights calculated!")

		for section, section_weight in section_weights.items():
			section_scores[section] = _block_ewq_score(section_weight)
		# 	print(f"Section {section}: {section_scores[section]}")
		# print("Section scores calculated!")
		# print("section_scores:", section_scores)

		###  module ewq score
		for module_idx, weights in module_weights.items():
			module_scores[module_idx] = _block_ewq_score(weights)
		# 	print(f"Module {module_idx}: {module_scores[module_idx]}")
		# print("Module scores calculated!")
		# print("module_scores:", module_scores)
		# module_scores = dict(sorted(module_scores.items(), key=lambda item: item[1]))
		# print("module_scores:", module_scores)

		# mu = np.mean(list(module_scores.values()))
		# print("mu:", mu)
		# sigma = np.std(list(module_scores.values()))

		# to_quantize = []
		# to_keep = []
		# for weight_name in layer_names:
		# 	idx = int(re.match(r'model\.(\d+)', weight_name).group(1))
		# 	if module_scores[idx] <= mu:
		# 		to_quantize.append(weight_name)
		# 	else:
		# 		to_keep.append(weight_name)
		# print("to_keep:", to_keep)
		# print("to_quantize:", to_quantize)

		# REMOVE LAYER EWQ SCORE

		### block + layer normalise before
		def normalize_dict(scores):

			min_val = min(scores.values())
			max_val = max(scores.values())

			return {
				k: (v - min_val) / (max_val + 1e-8)
				for k, v in scores.items()
			}
			
		# ### normalize before
		# module_scores	 = normalize_dict(module_scores)
		# layer_scores = _normalize_scores(layer_scores)

		# section ewq score
		# for layer_name, idx in zip(layer_names, range(len(layer_names))):
		# 	match = re.match(r'model\.(\d+)', layer_name)
		# 	layer_idx = int(match.group(1))
		# 	for section, section_idx in section_names.items():
		# 		if layer_idx <= section_idx:
		# 			layer_scores[idx]+=section_scores[section]
		# 			print(f"Layer {layer_name} (idx={layer_idx}){layer_scores[idx]} updated with section score {section_scores[section]} for section {section}")
		# 			break
		# print("Layer scores calculated!")

		##  module ewq score
		# for layer_name, idx in zip(layer_names, range(len(layer_names))):
		# 	match = re.match(r'model\.(\d+)', layer_name)
		# 	layer_idx = int(match.group(1))
		# 	for module_idx, module_score in module_scores.items():
		# 		if layer_idx == module_idx:
		# 			layer_scores[idx]+=module_score
		# 			print(f"Layer {layer_name} (idx={layer_idx}){layer_scores[idx]} updated with module score {module_score} for module {module_idx}")
		# 			break
		# print("Layer scores updated with module scores!")
				


		output_dir = "./results"
		model_name = os.path.basename(self.model_path.rstrip("/"))
		os.makedirs(output_dir, exist_ok=True)
		out_file = os.path.join(output_dir, f"{model_name}_ewq.json")
		result = {"scores": layer_scores, "layer_names": layer_names}
		with open(out_file, "w") as f:
			json.dump(result, f, indent=2)

		### calculate scores
		raw_scores = result.get("scores", [])
		layer_names = result.get("layer_names", [])
		# print(raw_scores)
		# print(layer_names)

		### normalize after 
		raw_scores = _normalize_scores(raw_scores)
		# print(raw_scores)

		ranked_nodes = []
		ranked_weights = []

		print(f"\n[*] Top-5 most sensitive layers (0-indexed):")
		ranked = _rank_layers(raw_scores)
		print("ranked:", ranked)
		for rank, layer_idx in enumerate(ranked): #[:5]
			print(f"    Rank {rank + 1}: Layer {layer_names[layer_idx].replace('.weight', '')}  (score={raw_scores[layer_idx]:.6f})")


		import random

		pairs = []

		for i, s1 in enumerate(raw_scores):
			for j, s2 in enumerate(raw_scores):
				li, lj = int(layer_names[i].split('.')[1]), int(layer_names[j].split('.')[1])

				if li <= 22 and lj == 23 and abs(s1 - s2) < 1e-6:
					print(f"{layer_names[i]} | {layer_names[j]} | {s1:.6f}")
					pairs.append((i, j))

		to_compare = []
		to_compare_weights = []

		if pairs:
			for i, j in random.sample(pairs, min(3, len(pairs))):

				to_compare_weights.append(layer_names[i])
				to_compare_weights.append(layer_names[j])
		else:
			print("no such nodes")
		for rank, layer_idx in enumerate(ranked):
			ranked_nodes.append(layer_names[layer_idx].replace('.weight', ''))
			ranked_weights.append(layer_names[layer_idx])

		print("ranked_nodes:", ranked_nodes[::-1])
		# ranked_weights.reverse()
		# print(ranked_weights)
		#### helper func
		
		
		total_param = sum(int(np.prod(init.dims)) for init in model.graph.initializer if "weight" in init.name and "conv" in init.name)
		print(total_param)
		# target_param = (1 - self.target) / 0.75 * total_param
		target_param = total_param 

		print("total_param:", total_param)

		inits = {init.name: init for init in model.graph.initializer if "weight" in init.name and "conv" in init.name}
		init_to_node = {}
		for node in model.graph.node:
			### init_to_node ref
			init_to_node.update({inp: node.name for inp in node.input if inp in inits})
		print(init_to_node)
		# to_quantize = [init_to_node[w] for w in to_quantize]
		# print("to_quantize nodes:", to_quantize)
		for rank, layer_idx in enumerate(ranked):
			node_name = init_to_node[layer_names[layer_idx]]
			weight_size = weight_params[layer_names[layer_idx]]
			arr.append([node_name, weight_size])
		print("arr:", arr)

		# to_compare = [init_to_node[w] for w in to_compare_weights]
		# print("to_compare:", to_compare)
	
		print(weight_params)
		selected, acc = [], 0
		for weight in ranked_weights:
			w = weight_params[weight]
			selected.append(init_to_node[weight])
			acc += w
			# print(f"acc: {acc}, target_param: {target_param}, selected: {selected}")
			if acc > target_param:
				acc -= w
				break

		# print(acc)
		# print(target_param)
		# print(len(ranked_nodes))
		# print(len(selected))
		selected
		# print(len(selected))
		# print(acc)
		# print(selected)
		return selected

In [5]:
if __name__ == '__main__':
	# from ultralytics import YOLO
	# model = YOLO("yolo11n-seg.pt")
	# model.export(format="onnx")
	
	ewq = ewq(target=0)
	result = ewq.filter()	

432
-2.1096385791207905e-05
432
4608
0.0009946910043557484
4608
1024
0.004402242600917816
1024
1152
0.003925221661726634
1152
1152
0.0012640605370203655
1152
3072
0.0014876794690887134
3072
36864
0.00012484967656847503
36864
4096
0.0011183300521224737
4096
4608
0.0009946989723377759
4608
4608
0.0009946119454171923
4608
12288
0.0003740837952742974
12288
147456
3.1226194955201616e-05
147456
16384
0.0002806959964800626
16384
2048
0.002225038595497608
2048
2048
0.002221919596195221
2048
9216
0.0004984964099195269
9216
9216
0.000498510172797574
9216
9216
0.0004985134841667281
9216
9216
0.00049850277395712
9216
4096
0.0011183046735823154
4096
24576
0.00018721791760375103
24576
294912
1.5614250312662785e-05
294912
65536
7.024560909485444e-05
65536
8192
0.0005606566555798054
8192
8192
0.0005605715559795499
8192
36864
0.00012484783979339732
36864
36864
0.00012484953428308168
36864
36864
0.00012484950841301016
36864
36864
0.00012484941786775985
36864
16384
0.0002807035925798118
16384
98304
4.683

C:\Users\Jenny\AppData\Local\Temp\ipykernel_30232\2434131981.py:33: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights = torch.from_numpy(W_np).float()
